# CrewAI Mini-Capstone Project: Restaurant Social Media Marketing Pipeline

## Scenario

**Spice Fusion Bistro** is launching a new menu of **Indian/Pakistani fusion dishes** and
wants a marketing blog post plus social media content to promote it.

You will build a **5-agent sequential CrewAI pipeline** that:

1. **Culinary Trend Researcher** — discovers the new fusion dishes on the menu
2. **Nutrition Analyst** — looks up nutrition facts for each dish
3. **Local Sourcing Specialist** — identifies which ingredients are locally/seasonally sourced
4. **Blog Content Writer** — synthesizes all of the above into a publish-ready blog post
5. **Social Media Strategist** — repurposes the blog post into a Facebook post and a Twitter/X thread

Each agent (except the last two) is backed by a **tool** that queries a small mock SQLite
database (`restaurant_marketing.db`) containing:

- `dishes` — the new fusion menu items
- `nutrition` — nutrition facts per dish
- `produce` — local/regional sourcing info per ingredient

This is a **pure CrewAI** project: no LangGraph, no `main.py`, no `crewai run`. Everything
runs top-to-bottom in this notebook via plain `Crew(...).kickoff()`.

**Note on formatting:** the blog and social media tasks are written to produce three
clearly labeled sections — `## Blog Post`, `## Facebook Post`, `## Twitter Thread` — even
though this notebook doesn't parse them apart programmatically. Labeling sections
explicitly like this is good practice regardless of whether something downstream reads
them: it makes the output easy for a human to scan, and it's the same pattern you'd rely
on if you later fed this output into another system (e.g. a LangGraph node) that *does*
need to split it apart.

## Assignment Notebook (fill in the TODOs)

## Section 1 — Environment Setup


In [ ]:
import os
import sqlite3

from crewai import Agent, Task, Crew, Process
from crewai.tools import tool

# TODO: Make sure your OPENAI_API_KEY (or other LLM provider key) is set in the
# environment before running this notebook, e.g.:
# os.environ["OPENAI_API_KEY"] = "sk-..."


## Section 2 — Mock Database Setup

We simulate the restaurant's internal systems with three SQLite tables:

- `dishes(id, name, cuisine, main_ingredients, description)`
- `nutrition(dish_id, calories, protein_g, carbs_g, fat_g, fiber_g, notes)`
- `produce(ingredient, region, season, local_farm_source)`


In [ ]:
DB_PATH = "restaurant_marketing.db"


def setup_database():
    """Create and populate the mock restaurant marketing database."""
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # TODO 1: Create the `dishes` table with columns:
    #   id (INTEGER PRIMARY KEY), name, cuisine, main_ingredients, description
    cursor.execute("""
        CREATE TABLE dishes (
            -- your columns here
        )
    """)

    # TODO 2: Create the `nutrition` table with columns:
    #   dish_id, calories, protein_g, carbs_g, fat_g, fiber_g, notes
    #   (dish_id should reference dishes.id)
    cursor.execute("""
        CREATE TABLE nutrition (
            -- your columns here
        )
    """)

    # TODO 3: Create the `produce` table with columns:
    #   ingredient, region, season, local_farm_source
    cursor.execute("""
        CREATE TABLE produce (
            -- your columns here
        )
    """)

    # TODO 4: Insert at least 5 new Indian/Pakistani fusion dishes into `dishes`.
    # Each row: (id, name, cuisine, main_ingredients, description)
    # Example idea: "Butter Chicken Tacos", "Lahori Chapli Burger", "Nihari Ramen", etc.
    dishes = [
        # (1, "Dish Name", "Fusion", "ingredient1, ingredient2, ...", "short description"),
    ]
    cursor.executemany("INSERT INTO dishes VALUES (?, ?, ?, ?, ?)", dishes)

    # TODO 5: Insert matching nutrition rows for each dish_id above.
    # Each row: (dish_id, calories, protein_g, carbs_g, fat_g, fiber_g, notes)
    nutrition = [
        # (1, 420, 28.0, 32.0, 19.0, 3.0, "short nutrition note"),
    ]
    cursor.executemany("INSERT INTO nutrition VALUES (?, ?, ?, ?, ?, ?, ?)", nutrition)

    # TODO 6: Insert at least 5 produce/sourcing rows.
    # Each row: (ingredient, region, season, local_farm_source)
    produce = [
        # ("Tomato", "Local Valley Farms", "Summer", "Green Acres Co-op"),
    ]
    cursor.executemany("INSERT INTO produce VALUES (?, ?, ?, ?)", produce)

    conn.commit()
    conn.close()
    print("Database setup complete.")


setup_database()


## Section 3 — Tools

Tools are how agents reach into the mock database. Each tool wraps a small, focused
SQL query and returns a plain-text summary the LLM can reason over.


In [ ]:
# TODO 7: Implement a tool that returns all rows from `dishes` as a readable
# text summary (name, cuisine, main_ingredients, description).
@tool("Trending Dishes Lookup")
def get_trending_dishes() -> str:
    """Returns the restaurant's new/trending Indian and Pakistani fusion dishes,
    including cuisine influence, main ingredients, and a short description."""
    # your implementation here
    pass


# TODO 8: Implement a tool that takes a dish_name string and returns its
# nutrition facts by joining `dishes` and `nutrition`. Use a case-insensitive
# partial match (LIKE) so "biryani" matches "Karachi Biryani Arancini".
@tool("Nutrition Info Lookup")
def get_nutrition_info(dish_name: str) -> str:
    """Given a dish name (or partial name), returns its nutrition facts:
    calories, protein, carbs, fat, and fiber."""
    # your implementation here
    pass


# TODO 9: Implement a tool that returns all rows from `produce` as a readable
# text summary (ingredient, region, season, local_farm_source).
@tool("Local Produce Sourcing Lookup")
def get_local_produce_info() -> str:
    """Returns sourcing details for key ingredients: region, season, and
    local farm/supplier."""
    # your implementation here
    pass


## Section 4 — Agents

Five agents, each with a narrow role, a clear goal, and a backstory that shapes tone.
Only the first three need tools — the writer and strategist work purely from the
context passed to them by earlier tasks.


In [ ]:
# TODO 10: Define the Culinary Trend Researcher agent.
# - role: "Culinary Trend Researcher"
# - goal: identify/describe the restaurant's new fusion dishes
# - backstory: a food journalist persona (your own wording)
# - tools: [get_trending_dishes]
trend_researcher = Agent(
    role="",
    goal="",
    backstory="",
    tools=[],
    verbose=True,
)

# TODO 11: Define the Nutrition Analyst agent.
# - role: "Nutrition Analyst"
# - goal: provide friendly, accurate nutrition info per dish
# - backstory: a registered dietitian persona
# - tools: [get_nutrition_info]
nutrition_analyst = Agent(
    role="",
    goal="",
    backstory="",
    tools=[],
    verbose=True,
)

# TODO 12: Define the Local Sourcing Specialist agent.
# - role: "Local Sourcing Specialist"
# - goal: highlight local/seasonal ingredient sourcing
# - backstory: a supply-chain coordinator persona
# - tools: [get_local_produce_info]
sourcing_specialist = Agent(
    role="",
    goal="",
    backstory="",
    tools=[],
    verbose=True,
)

# TODO 13: Define the Blog Content Writer agent.
# - role: "Blog Content Writer"
# - goal: synthesize dish, nutrition, and sourcing info into a blog post
# - backstory: a restaurant marketing copywriter persona
# - NOTE: this agent needs no tools -- it works from task context only
blog_writer = Agent(
    role="",
    goal="",
    backstory="",
    verbose=True,
)

# TODO 14: Define the Social Media Strategist agent.
# - role: "Social Media Strategist"
# - goal: repurpose the blog post into Facebook + Twitter/X content
# - backstory: a restaurant social media manager persona
# - NOTE: this agent needs no tools -- it works from task context only
social_media_strategist = Agent(
    role="",
    goal="",
    backstory="",
    verbose=True,
)


## Section 5 — Tasks

Tasks are chained with `context=[...]` so each downstream task receives the outputs
of the tasks it depends on. This is the same context-passing pattern used in the
bond trading pipeline (market data -> position -> risk -> threshold -> decision).


In [ ]:
# TODO 15: Define research_task.
# - description: ask the Culinary Trend Researcher to find the new fusion
#   dishes and capture name, cuisine influence, main ingredients, and hook.
# - expected_output: a list of dishes with a short hook for each.
# - agent: trend_researcher
research_task = Task(
    description="",
    expected_output="",
    agent=trend_researcher,
)

# TODO 16: Define nutrition_task.
# - description: ask the Nutrition Analyst to look up nutrition facts for
#   each dish from research_task and summarize them in friendly language.
# - expected_output: a nutrition summary per dish.
# - agent: nutrition_analyst
# - context: [research_task]  <-- this is how downstream tasks see upstream output
nutrition_task = Task(
    description="",
    expected_output="",
    agent=nutrition_analyst,
    context=[],
)

# TODO 17: Define sourcing_task.
# - description: ask the Local Sourcing Specialist to identify local/seasonal
#   sourcing for the ingredients in the dishes from research_task.
# - expected_output: a short sourcing write-up per dish/ingredient.
# - agent: sourcing_specialist
# - context: [research_task]
sourcing_task = Task(
    description="",
    expected_output="",
    agent=sourcing_specialist,
    context=[],
)

# TODO 18: Define blog_task.
# - description: ask the Blog Content Writer to combine everything into a
#   600-900 word blog post with an intro, one section per dish, and a
#   closing call-to-action. Put the whole post under the exact Markdown
#   heading '## Blog Post' (good practice for making output easy to parse
#   downstream, even though this notebook doesn't parse it itself).
# - expected_output: a publish-ready Markdown blog post starting with that heading.
# - agent: blog_writer
# - context: [research_task, nutrition_task, sourcing_task]  <-- all three feed the writer
blog_task = Task(
    description="",
    expected_output="",
    agent=blog_writer,
    context=[],
)

# TODO 19: Define social_media_task.
# - description: ask the Social Media Strategist to produce TWO clearly
#   separated sections: a Facebook post (150-200 words + hashtags) under the
#   exact heading '## Facebook Post', and a 4-5 tweet Twitter/X thread under
#   the exact heading '## Twitter Thread', ending with a call-to-action.
# - expected_output: a labeled Facebook post + Twitter/X thread.
# - agent: social_media_strategist
# - context: [blog_task]
social_media_task = Task(
    description="",
    expected_output="",
    agent=social_media_strategist,
    context=[],
)


## Section 6 — Assemble and Run the Crew


In [ ]:
# TODO 20: Assemble the Crew.
# - agents: all 5 agents, in the same order they run
# - tasks: all 5 tasks, in the same order they run
# - process: Process.sequential
crew = Crew(
    agents=[],
    tasks=[],
    process=None,
    verbose=True,
)

# TODO 21: Kick off the crew and store the result.
result = None


## Section 7 — Review the Output

Run the cell below after `kickoff()` completes to inspect the final deliverable:
the blog post and social media content produced by the last task in the pipeline.


In [ ]:
print(result)


## Reflection Questions

1. Why does `blog_task` list **three** tasks in its `context`, while `nutrition_task`
   and `sourcing_task` each list only **one**?
2. The final task is instructed to use exact headings ('## Facebook Post',
   '## Twitter Thread') even though nothing in this notebook parses them apart
   programmatically. Why might that formatting discipline still be worth
   enforcing? What would you need to change if you later wanted to split the
   Facebook and Twitter sections into separate variables?
3. `Process.sequential` runs tasks strictly in order. Sketch (in words, no code
   needed) how you'd restructure this pipeline using `Process.hierarchical` with a
   manager agent instead. What would the manager decide?
4. The nutrition tool uses a `LIKE` match on dish name. What's a failure case where
   this could return the wrong dish's nutrition info, and how would you fix it?
